In [1]:
"""
Diagnose all FSL dynamic-sign datasets and optionally trained models.

Checks:
- Dataset structure
- Class distribution
- Train/val/test distribution
- Missing classes
- Sequence lengths
- Feature dimensions
- Zero/padded frames
- Landmark detection rate
- Left/right/both-hand detection
- Per-class statistics
- Optional model evaluation
- Confusion matrix
- Per-class accuracy

Usage:
    python diagnose_all_models.py

Adjust MODEL_CONFIGS below to match your project.
"""

import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch


# ============================================================
# PATH
# ============================================================

REPO_ROOT = Path(
    r"C:\Projects\signia-fsl-recognition"
).resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


# ============================================================
# CONFIG
# ============================================================

DATASET_PATH = (
    REPO_ROOT
    / "notebooks"
    / "02_dynamic"
    / "fsl_dataset_clean.pt"
)

LABELS_CSV = (
    REPO_ROOT
    / "csv"
    / "labels.csv"
)


# ------------------------------------------------------------
# Models to diagnose
#
# Change label_ids/output/model paths as needed.
# ------------------------------------------------------------

MODEL_CONFIGS = {

    "greetings": {
        "label_ids": list(range(10)),
        "model_path": (
            REPO_ROOT
            / "artifacts"
            / "models"
            / "greeting"
            / "greetings_lstm_best.pt"
        ),
        "model_type": "sign_lstm",
        "input_size": 252,
        "num_classes": 10,
    },

    "survival": {
        "label_ids": list(range(10)),
        "model_path": (
            REPO_ROOT
            / "artifacts"
            / "models"
            / "survival"
            / "survival_lstm_best.pt"
        ),
        "model_type": "sign_lstm",
        "input_size": 252,
        "num_classes": 10,
    },

    "numbers": {
        "label_ids": list(range(10)),
        "model_path": (
            REPO_ROOT
            / "artifacts"
            / "models"
            / "number"
            / "numbers_lstm_best.pt"
        ),
        "model_type": "sign_lstm",
        "input_size": 252,
        "num_classes": 10,
    },

    "calendar": {
        "label_ids": list(range(12)),
        "model_path": (
            REPO_ROOT
            / "artifacts"
            / "models"
            / "calendar"
            / "calendar_lstm_best.pt"
        ),
        "model_type": "sign_lstm",
        "input_size": 252,
        "num_classes": 12,
    },
}


# ============================================================
# HELPERS
# ============================================================

def print_header(title):

    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def print_section(title):

    print()
    print("-" * 70)
    print(title)
    print("-" * 70)


def get_label_names(labels_csv):

    """
    Read labels.csv if possible.

    Expected common formats:
        id,label
        0,HELLO

    Adapt this function if your CSV uses different columns.
    """

    import csv

    labels = {}

    if not labels_csv.exists():

        print(
            f"WARNING: labels CSV not found: {labels_csv}"
        )

        return labels

    with open(
        labels_csv,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            # Try common column names.

            id_key = None
            label_key = None

            for key in row:

                lower = key.lower().strip()

                if lower in {
                    "id",
                    "label_id",
                    "class_id"
                }:
                    id_key = key

                if lower in {
                    "label",
                    "name",
                    "class",
                    "class_name"
                }:
                    label_key = key

            if id_key is None or label_key is None:
                continue

            try:

                label_id = int(row[id_key])

                labels[label_id] = (
                    row[label_key]
                    .strip()
                )

            except (
                ValueError,
                TypeError
            ):
                continue

    return labels


def label_name(label_id, label_names):

    return label_names.get(
        label_id,
        f"UNKNOWN_{label_id}"
    )


def is_frame_detected(frame):

    """
    A frame is considered detected if it contains
    at least one non-zero landmark value.
    """

    return np.any(frame != 0)


def get_hand_detection(frame):

    """
    Assumes:

        first 63  = left hand
        next 63   = right hand

    Returns:
        left_detected
        right_detected
    """

    if frame.shape[-1] < 126:

        return False, False

    left = frame[:63]

    right = frame[63:126]

    left_detected = np.any(left != 0)

    right_detected = np.any(right != 0)

    return (
        bool(left_detected),
        bool(right_detected)
    )


# ============================================================
# DATASET LOADING
# ============================================================

def load_dataset(path):

    print_header("LOADING DATASET")

    print("Path:")
    print(path)

    if not path.exists():

        raise FileNotFoundError(
            f"Dataset not found: {path}"
        )

    dataset = torch.load(
        path,
        map_location="cpu"
    )

    print(
        "\nDataset type:",
        type(dataset)
    )

    if isinstance(dataset, dict):

        print(
            "\nDataset keys:"
        )

        for key in dataset.keys():

            value = dataset[key]

            if hasattr(value, "shape"):

                print(
                    f"  {key}: "
                    f"shape={value.shape}"
                )

            else:

                print(
                    f"  {key}: "
                    f"type={type(value)}"
                )

    return dataset


# ============================================================
# DATASET STRUCTURE
# ============================================================

def find_dataset_arrays(dataset):

    """
    Try to identify features and labels from common
    dictionary formats.

    Modify this if your dataset uses different keys.
    """

    if not isinstance(dataset, dict):

        raise ValueError(
            "Expected dataset to be a dictionary."
        )

    feature_keys = [
        "X",
        "x",
        "features",
        "sequences",
        "data",
        "inputs",
    ]

    label_keys = [
        "y",
        "labels",
        "targets",
        "target",
    ]

    X = None
    y = None

    for key in feature_keys:

        if key in dataset:

            X = dataset[key]

            break

    for key in label_keys:

        if key in dataset:

            y = dataset[key]

            break

    if X is None:

        raise KeyError(
            "Could not find feature array in dataset."
        )

    if y is None:

        raise KeyError(
            "Could not find label array in dataset."
        )

    if isinstance(X, torch.Tensor):

        X = X.cpu().numpy()

    if isinstance(y, torch.Tensor):

        y = y.cpu().numpy()

    return X, y


# ============================================================
# BASIC DATASET STATS
# ============================================================

def diagnose_basic(X, y, label_names):

    print_header("BASIC DATASET INFORMATION")

    print(
        "Feature shape:",
        X.shape
    )

    print(
        "Label shape:",
        y.shape
    )

    print(
        "Number of samples:",
        len(X)
    )

    if X.ndim >= 2:

        print(
            "Sequence length:",
            X.shape[1]
        )

    if X.ndim >= 3:

        print(
            "Features per frame:",
            X.shape[2]
        )

    print(
        "Number of unique labels:",
        len(np.unique(y))
    )

    print_section("CLASS DISTRIBUTION")

    counts = Counter(
        y.tolist()
    )

    for label_id in sorted(counts):

        print(
            f"{label_id:4d}  "
            f"{label_name(label_id, label_names):25s} "
            f"{counts[label_id]:5d}"
        )


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

def diagnose_expected_classes(
    y,
    label_ids,
    label_names
):

    print_section(
        "EXPECTED CLASS COVERAGE"
    )

    counts = Counter(
        y.tolist()
    )

    for label_id in label_ids:

        count = counts.get(
            label_id,
            0
        )

        status = "OK"

        if count == 0:

            status = "MISSING"

        elif count < 5:

            status = "VERY LOW"

        elif count < 10:

            status = "LOW"

        print(
            f"{label_id:4d}  "
            f"{label_name(label_id, label_names):25s} "
            f"{count:5d}  "
            f"{status}"
        )


# ============================================================
# SEQUENCE QUALITY
# ============================================================

def diagnose_sequences(
    X,
    y,
    label_names
):

    print_section(
        "SEQUENCE / LANDMARK QUALITY"
    )

    if X.ndim != 3:

        print(
            "WARNING: Expected "
            "(samples, frames, features)."
        )

        return

    per_class = defaultdict(
        lambda: {
            "samples": 0,
            "total_frames": 0,
            "detected_frames": 0,
            "left": 0,
            "right": 0,
            "both": 0,
            "none": 0,
        }
    )

    for sequence, label in zip(
        X,
        y
    ):

        stats = per_class[
            int(label)
        ]

        stats["samples"] += 1

        for frame in sequence:

            stats["total_frames"] += 1

            left, right = get_hand_detection(
                frame
            )

            if left or right:

                stats[
                    "detected_frames"
                ] += 1

            if left:
                stats["left"] += 1

            if right:
                stats["right"] += 1

            if left and right:
                stats["both"] += 1

            if not left and not right:
                stats["none"] += 1

    print(
        f"{'Class':25s} "
        f"{'Samples':>7s} "
        f"{'Detect':>8s} "
        f"{'Left':>8s} "
        f"{'Right':>8s} "
        f"{'Both':>8s} "
        f"{'None':>8s}"
    )

    print("-" * 90)

    for label_id in sorted(per_class):

        stats = per_class[label_id]

        total = stats["total_frames"]

        detection_rate = (
            stats["detected_frames"]
            / total
            * 100
            if total > 0
            else 0
        )

        left_rate = (
            stats["left"]
            / total
            * 100
            if total > 0
            else 0
        )

        right_rate = (
            stats["right"]
            / total
            * 100
            if total > 0
            else 0
        )

        both_rate = (
            stats["both"]
            / total
            * 100
            if total > 0
            else 0
        )

        none_rate = (
            stats["none"]
            / total
            * 100
            if total > 0
            else 0
        )

        print(
            f"{label_name(label_id, label_names):25s} "
            f"{stats['samples']:7d} "
            f"{detection_rate:7.1f}% "
            f"{left_rate:7.1f}% "
            f"{right_rate:7.1f}% "
            f"{both_rate:7.1f}% "
            f"{none_rate:7.1f}%"
        )


# ============================================================
# ZERO FRAME ANALYSIS
# ============================================================

def diagnose_zero_frames(
    X,
    y,
    label_names
):

    print_section(
        "ZERO / EMPTY FRAME ANALYSIS"
    )

    for label_id in sorted(
        np.unique(y)
    ):

        class_sequences = X[
            y == label_id
        ]

        total_frames = (
            class_sequences.shape[0]
            * class_sequences.shape[1]
        )

        zero_frames = 0

        for sequence in class_sequences:

            for frame in sequence:

                if not np.any(frame != 0):

                    zero_frames += 1

        rate = (
            zero_frames
            / total_frames
            * 100
            if total_frames > 0
            else 0
        )

        status = ""

        if rate > 20:
            status = "  ⚠ HIGH"

        elif rate > 5:
            status = "  ⚠"

        print(
            f"{label_name(label_id, label_names):25s} "
            f"{zero_frames:6d}/"
            f"{total_frames:<6d} "
            f"({rate:6.2f}%){status}"
        )


# ============================================================
# SPLIT DIAGNOSTICS
# ============================================================

def diagnose_splits(
    dataset,
    label_names
):

    print_section(
        "TRAIN / VALIDATION / TEST SPLITS"
    )

    possible_splits = [
        ("train", "train_labels"),
        ("val", "val_labels"),
        ("validation", "validation_labels"),
        ("test", "test_labels"),
    ]

    found = False

    for split_name, label_key in possible_splits:

        if label_key not in dataset:

            continue

        found = True

        labels = dataset[label_key]

        if isinstance(
            labels,
            torch.Tensor
        ):

            labels = labels.cpu().numpy()

        counts = Counter(
            labels.tolist()
        )

        print(
            f"\n{split_name.upper()}: "
            f"{len(labels)} samples"
        )

        for label_id in sorted(counts):

            print(
                f"  {label_name(label_id, label_names):25s}"
                f"{counts[label_id]:5d}"
            )

        # Detect missing test classes.

        if split_name == "test":

            missing = [
                label_id
                for label_id in label_names
                if counts.get(
                    label_id,
                    0
                ) == 0
            ]

            if missing:

                print(
                    "\n  ⚠ TEST SET MISSING CLASSES:"
                )

                for label_id in missing:

                    print(
                        f"    - "
                        f"{label_name(label_id, label_names)}"
                    )

    if not found:

        print(
            "No explicit train/val/test labels "
            "found in dataset."
        )


# ============================================================
# WARNINGS
# ============================================================

def generate_warnings(
    X,
    y,
    label_ids,
    label_names
):

    print_header(
        "AUTOMATIC DIAGNOSTIC WARNINGS"
    )

    warnings = []

    counts = Counter(
        y.tolist()
    )

    for label_id in label_ids:

        count = counts.get(
            label_id,
            0
        )

        name = label_name(
            label_id,
            label_names
        )

        if count == 0:

            warnings.append(
                f"{name}: no samples found"
            )

        elif count < 10:

            warnings.append(
                f"{name}: only {count} samples"
            )

    # --------------------------------------------------------
    # Detection rate
    # --------------------------------------------------------

    if X.ndim == 3 and X.shape[-1] >= 126:

        for label_id in sorted(
            np.unique(y)
        ):

            sequences = X[
                y == label_id
            ]

            total = (
                sequences.shape[0]
                * sequences.shape[1]
            )

            detected = 0

            for sequence in sequences:

                for frame in sequence:

                    if np.any(
                        frame != 0
                    ):

                        detected += 1

            rate = (
                detected / total * 100
                if total
                else 0
            )

            if rate < 70:

                warnings.append(
                    f"{label_name(label_id, label_names)}: "
                    f"low landmark detection "
                    f"({rate:.1f}%)"
                )

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    if not warnings:

        print(
            "No major automatic warnings."
        )

    else:

        for warning in warnings:

            print(
                f"⚠ {warning}"
            )


# ============================================================
# MODEL EVALUATION
# ============================================================

def evaluate_model(
    model,
    X,
    y,
    label_ids,
    label_names
):

    print_header(
        "MODEL EVALUATION"
    )

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.long
    )

    correct = Counter()
    total = Counter()

    predictions = []

    with torch.no_grad():

        batch_size = 32

        for start in range(
            0,
            len(X_tensor),
            batch_size
        ):

            batch = X_tensor[
                start:start + batch_size
            ]

            targets = y_tensor[
                start:start + batch_size
            ]

            outputs = model(
                batch
            )

            predicted = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                predicted.cpu().tolist()
            )

            for true, pred in zip(
                targets.tolist(),
                predicted.tolist()
            ):

                total[true] += 1

                if true == pred:

                    correct[true] += 1

    print(
        f"{'Class':25s} "
        f"{'Correct':>8s} "
        f"{'Total':>8s} "
        f"{'Accuracy':>10s}"
    )

    print("-" * 60)

    for label_id in label_ids:

        c = correct[label_id]

        t = total[label_id]

        if t == 0:

            accuracy = float("nan")

        else:

            accuracy = (
                c / t * 100
            )

        print(
            f"{label_name(label_id, label_names):25s} "
            f"{c:8d} "
            f"{t:8d} "
            f"{accuracy:9.2f}%"
        )

    return np.array(
        predictions
    )


# ============================================================
# CONFUSION MATRIX
# ============================================================

def print_confusion_matrix(
    y_true,
    y_pred,
    label_ids,
    label_names
):

    print_header(
        "CONFUSION MATRIX"
    )

    label_ids = list(
        label_ids
    )

    matrix = np.zeros(
        (
            len(label_ids),
            len(label_ids)
        ),
        dtype=int
    )

    id_to_index = {
        label_id: i
        for i, label_id
        in enumerate(label_ids)
    }

    for true, pred in zip(
        y_true,
        y_pred
    ):

        if (
            true in id_to_index
            and pred in id_to_index
        ):

            matrix[
                id_to_index[true],
                id_to_index[pred]
            ] += 1

    names = [
        label_name(
            label_id,
            label_names
        )
        for label_id in label_ids
    ]

    short_names = [
        name[:12]
        for name in names
    ]

    print(
        "\nActual \\ Predicted"
    )

    print(
        f"{'':18s}",
        end=""
    )

    for name in short_names:

        print(
            f"{name:>13s}",
            end=""
        )

    print()

    for i, name in enumerate(
        short_names
    ):

        print(
            f"{name:<18s}",
            end=""
        )

        for j in range(
            len(label_ids)
        ):

            print(
                f"{matrix[i, j]:13d}",
                end=""
            )

        print()


# ============================================================
# MODEL LOADING
# ============================================================

def load_model(config):

    model_path = config.get(
        "model_path"
    )

    if model_path is None:

        return None

    if not model_path.exists():

        print(
            f"\nModel not found:"
            f"\n{model_path}"
        )

        return None

    print(
        f"\nLoading model:"
        f"\n{model_path}"
    )

    try:

        from src.models.sign_lstm import (
            SignLSTM
        )

        model = SignLSTM(
            input_size=config[
                "input_size"
            ],
            hidden_size=128,
            num_layers=2,
            num_classes=config[
                "num_classes"
            ],
        )

        checkpoint = torch.load(
            model_path,
            map_location="cpu"
        )

        model.load_state_dict(
            checkpoint
        )

        model.eval()

        print(
            "Model loaded successfully."
        )

        return model

    except Exception as e:

        print(
            "\nWARNING: Could not load model."
        )

        print(
            type(e).__name__,
            str(e)
        )

        return None


# ============================================================
# RUN ONE MODEL
# ============================================================

def diagnose_model(
    model_name,
    config,
    dataset,
    label_names
):

    print_header(
        f"DIAGNOSING: {model_name.upper()}"
    )

    X, y = find_dataset_arrays(
        dataset
    )

    label_ids = config[
        "label_ids"
    ]

    diagnose_basic(
        X,
        y,
        label_names
    )

    diagnose_expected_classes(
        y,
        label_ids,
        label_names
    )

    diagnose_sequences(
        X,
        y,
        label_names
    )

    diagnose_zero_frames(
        X,
        y,
        label_names
    )

    diagnose_splits(
        dataset,
        label_names
    )

    generate_warnings(
        X,
        y,
        label_ids,
        label_names
    )

    # --------------------------------------------------------
    # Optional model evaluation
    # --------------------------------------------------------

    model = load_model(
        config
    )

    if model is not None:

        predictions = evaluate_model(
            model,
            X,
            y,
            label_ids,
            label_names
        )

        print_confusion_matrix(
            y,
            predictions,
            label_ids,
            label_names
        )


# ============================================================
# MAIN
# ============================================================

def main():

    print_header(
        "FSL MODEL / DATASET DIAGNOSTIC"
    )

    print(
        "Repository:",
        REPO_ROOT
    )

    print(
        "Dataset:",
        DATASET_PATH
    )

    label_names = get_label_names(
        LABELS_CSV
    )

    dataset = load_dataset(
        DATASET_PATH
    )

    for model_name, config in (
        MODEL_CONFIGS.items()
    ):

        try:

            diagnose_model(
                model_name,
                config,
                dataset,
                label_names
            )

        except Exception as e:

            print(
                f"\nERROR diagnosing "
                f"{model_name}:"
            )

            print(
                type(e).__name__
            )

            print(
                str(e)
            )

            import traceback

            traceback.print_exc()

    print_header(
        "DIAGNOSTICS COMPLETE"
    )


if __name__ == "__main__":

    main()


FSL MODEL / DATASET DIAGNOSTIC
Repository: C:\Projects\signia-fsl-recognition
Dataset: C:\Projects\signia-fsl-recognition\notebooks\02_dynamic\fsl_dataset_clean.pt

LOADING DATASET
Path:
C:\Projects\signia-fsl-recognition\notebooks\02_dynamic\fsl_dataset_clean.pt

Dataset type: <class 'dict'>

Dataset keys:
  X: shape=torch.Size([2129, 30, 126])
  y: shape=torch.Size([2129])

DIAGNOSING: GREETINGS

BASIC DATASET INFORMATION
Feature shape: (2129, 30, 126)
Label shape: (2129,)
Number of samples: 2129
Sequence length: 30
Features per frame: 126
Number of unique labels: 105

----------------------------------------------------------------------
CLASS DISTRIBUTION
----------------------------------------------------------------------
   0  GOOD MORNING                 20
   1  GOOD AFTERNOON               21
   2  GOOD EVENING                 22
   3  HELLO                        20
   4  HOW ARE YOU                  21
   5  IM FINE                      20
   6  NICE TO MEET YOU          

In [2]:
# scripts/diagnose_models.py

import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from src.models.factory import create_model
from src.training.label_encoder import ModelLabelEncoder


# ============================================================
# CONFIG
# ============================================================

REPO_ROOT = Path(
    r"C:\Projects\signia-fsl-recognition"
).resolve()

DATASET_PATH = (
    REPO_ROOT
    / "notebooks"
    / "02_dynamic"
    / "fsl_dataset_clean.pt"
)

LABELS_CSV = (
    REPO_ROOT
    / "csv"
    / "labels.csv"
)

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42


# Add/change paths here.
MODELS = {

    "greeting": {
        "label_ids": list(range(0, 10)),
        "model": (
            REPO_ROOT
            / "artifacts/models/greeting/"
            / "greetings_lstm_best.pt"
        ),
    },

    "survival": {
        "label_ids": list(range(10, 20)),
        "model": (
            REPO_ROOT
            / "artifacts/models/survival/"
            / "survival_lstm_best.pt"
        ),
    },

    "number": {
        "label_ids": list(range(20, 30)),
        "model": (
            REPO_ROOT
            / "artifacts/models/number/"
            / "numbers_lstm_best.pt"
        ),
    },

    "calendar": {
        "label_ids": list(range(30, 42)),
        "model": (
            REPO_ROOT
            / "artifacts/models/calendar/"
            / "calendar_lstm_best.pt"
        ),
    },
}


# ============================================================
# LOAD LABELS
# ============================================================

def load_labels():

    import pandas as pd

    df = pd.read_csv(LABELS_CSV)

    return {
        int(row.id): str(row.label)
        for _, row in df.iterrows()
    }


# ============================================================
# LOAD DATASET
# ============================================================

def load_dataset():

    bundle = torch.load(
        DATASET_PATH,
        map_location="cpu",
        weights_only=False,
    )

    X = np.asarray(bundle["X"])
    y = np.asarray(bundle["y"]).reshape(-1)

    return X, y


# ============================================================
# SPLIT — EXACT COPY OF TRAINER
# ============================================================

def split_dataset(X, y):

    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=(
            TEST_SIZE +
            VALIDATION_SIZE
        ),
        random_state=RANDOM_STATE,
        stratify=y,
    )

    validation_ratio = (
        VALIDATION_SIZE /
        (
            TEST_SIZE +
            VALIDATION_SIZE
        )
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=1 - validation_ratio,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
    )


# ============================================================
# LANDMARK QUALITY
# ============================================================

def landmark_quality(X, y, label_names):

    print("\nLANDMARK QUALITY")
    print("-" * 90)

    for label_id in sorted(np.unique(y)):

        samples = X[y == label_id]

        total = samples.size

        zeros = np.sum(
            samples == 0
        )

        zero_pct = (
            zeros / total * 100
        )

        # 63 left + 63 right
        left = samples[:, :, :63]
        right = samples[:, :, 63:]

        left_detected = np.any(
            left != 0,
            axis=2
        )

        right_detected = np.any(
            right != 0,
            axis=2
        )

        both = (
            left_detected &
            right_detected
        )

        detected = (
            left_detected |
            right_detected
        )

        print(
            f"{label_names.get(label_id, str(label_id)):25s}"
            f" samples={len(samples):3d}"
            f" detect={detected.mean()*100:6.1f}%"
            f" left={left_detected.mean()*100:6.1f}%"
            f" right={right_detected.mean()*100:6.1f}%"
            f" both={both.mean()*100:6.1f}%"
            f" zero={zero_pct:6.2f}%"
        )


# ============================================================
# MODEL
# ============================================================

def load_model(model_path):

    checkpoint = torch.load(
        model_path,
        map_location="cpu",
        weights_only=False,
    )

    if "model_state_dict" not in checkpoint:
        raise ValueError(
            "Checkpoint does not contain "
            "'model_state_dict'."
        )

    input_size = checkpoint["input_size"]
    num_classes = checkpoint["num_classes"]

    model = create_model(
        model_type="sign_lstm",
        num_classes=num_classes,
        input_size=input_size,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    return model, checkpoint


# ============================================================
# EVALUATE
# ============================================================

def evaluate_model(
    model,
    X_test,
    y_test,
    label_ids,
    label_names,
):

    X_tensor = torch.tensor(
        X_test,
        dtype=torch.float32,
    )

    with torch.no_grad():

        outputs = model(
            X_tensor
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        ).numpy()

    accuracy = (
        predictions == y_test
    ).mean()

    # --------------------------------------------------------
    # Per-class accuracy
    # --------------------------------------------------------

    print("\nPER-CLASS TEST ACCURACY")
    print("-" * 70)

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=range(len(label_ids)),
    )

    results = []

    for encoded_id, global_id in enumerate(
        label_ids
    ):

        mask = (
            y_test == encoded_id
        )

        total = mask.sum()

        if total == 0:
            continue

        correct = (
            predictions[mask] == encoded_id
        ).sum()

        class_acc = (
            correct / total
        )

        results.append(
            (
                class_acc,
                global_id,
                correct,
                total,
            )
        )

        print(
            f"{label_names.get(global_id, str(global_id)):25s}"
            f" {correct:2d}/{total:2d}"
            f" = {class_acc*100:6.2f}%"
        )

    # --------------------------------------------------------
    # Confusions
    # --------------------------------------------------------

    print("\nTOP CONFUSIONS")
    print("-" * 70)

    confusions = []

    for true_id in range(
        len(label_ids)
    ):

        for pred_id in range(
            len(label_ids)
        ):

            if true_id == pred_id:
                continue

            count = cm[
                true_id,
                pred_id
            ]

            if count > 0:

                confusions.append(
                    (
                        count,
                        label_ids[true_id],
                        label_ids[pred_id],
                    )
                )

    confusions.sort(
        reverse=True
    )

    for count, true_id, pred_id in confusions[:10]:

        print(
            f"{label_names.get(true_id, str(true_id))}"
            f" -> "
            f"{label_names.get(pred_id, str(pred_id))}"
            f" : {count}"
        )

    # --------------------------------------------------------
    # Worst classes
    # --------------------------------------------------------

    print("\nWORST CLASSES")
    print("-" * 70)

    for acc, global_id, correct, total in sorted(
        results
    )[:5]:

        print(
            f"{label_names.get(global_id, str(global_id)):25s}"
            f" {correct}/{total}"
            f" ({acc*100:.2f}%)"
        )

    return accuracy


# ============================================================
# DIAGNOSE MODEL
# ============================================================

def diagnose_model(
    name,
    model_config,
    X,
    y,
    label_names,
):

    print("\n")
    print("=" * 90)
    print(f"MODEL: {name.upper()}")
    print("=" * 90)

    label_ids = model_config[
        "label_ids"
    ]

    model_path = model_config[
        "model"
    ]

    # --------------------------------------------------------
    # Select classes
    # --------------------------------------------------------

    mask = np.isin(
        y,
        label_ids
    )

    X_selected = X[mask]
    y_global = y[mask]

    print(
        f"Samples: {len(X_selected)}"
    )

    print(
        f"Input shape: {X_selected.shape}"
    )

    print(
        f"Classes: {len(label_ids)}"
    )

    # --------------------------------------------------------
    # Distribution
    # --------------------------------------------------------

    print("\nCLASS DISTRIBUTION")
    print("-" * 70)

    for label_id in label_ids:

        count = np.sum(
            y_global == label_id
        )

        print(
            f"{label_id:3d} "
            f"{label_names.get(label_id, str(label_id)):25s}"
            f" {count}"
        )

    # --------------------------------------------------------
    # Landmark quality
    # --------------------------------------------------------

    landmark_quality(
        X_selected,
        y_global,
        label_names,
    )

    # --------------------------------------------------------
    # Encode labels exactly like trainer
    # --------------------------------------------------------

    encoder = ModelLabelEncoder()

    encoder.fit(
        sorted(
            np.unique(y_global)
        )
    )

    y_encoded = encoder.transform(
        y_global
    )

    # --------------------------------------------------------
    # Split exactly like trainer
    # --------------------------------------------------------

    (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
    ) = split_dataset(
        X_selected,
        y_encoded,
    )

    print("\nSPLIT")
    print("-" * 70)

    print(
        f"Train:      {len(X_train)}"
    )

    print(
        f"Validation: {len(X_val)}"
    )

    print(
        f"Test:       {len(X_test)}"
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    if not model_path.exists():

        print(
            f"\nMODEL NOT FOUND:\n{model_path}"
        )

        return

    model, checkpoint = load_model(
        model_path
    )

    print("\nCHECKPOINT")
    print("-" * 70)

    print(
        f"Input size: "
        f"{checkpoint.get('input_size')}"
    )

    print(
        f"Classes: "
        f"{checkpoint.get('num_classes')}"
    )

    print(
        f"Best val accuracy: "
        f"{checkpoint.get('best_val_accuracy')}"
    )

    print(
        f"Dataset input size: "
        f"{X_selected.shape[-1]}"
    )

    if (
        checkpoint["input_size"]
        != X_selected.shape[-1]
    ):

        print(
            "\nWARNING:"
            " checkpoint input size does not "
            "match dataset input size."
        )

        return

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    accuracy = evaluate_model(
        model,
        X_test,
        y_test,
        label_ids,
        label_names,
    )

    print(
        "\nOVERALL TEST ACCURACY:"
        f" {accuracy*100:.2f}%"
    )


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 90)
    print("FSL MODEL / DATASET DIAGNOSTIC")
    print("=" * 90)

    print(
        f"\nRepository:\n{REPO_ROOT}"
    )

    print(
        f"\nDataset:\n{DATASET_PATH}"
    )

    label_names = load_labels()

    X, y = load_dataset()

    print("\nDATASET")
    print("-" * 70)

    print(
        f"X shape: {X.shape}"
    )

    print(
        f"y shape: {y.shape}"
    )

    print(
        f"Samples: {len(X)}"
    )

    print(
        f"Unique labels: "
        f"{len(np.unique(y))}"
    )

    for name, config in MODELS.items():

        diagnose_model(
            name,
            config,
            X,
            y,
            label_names,
        )

    print("\n")
    print("=" * 90)
    print("DIAGNOSTIC COMPLETE")
    print("=" * 90)


if __name__ == "__main__":
    main()

FSL MODEL / DATASET DIAGNOSTIC

Repository:
C:\Projects\signia-fsl-recognition

Dataset:
C:\Projects\signia-fsl-recognition\notebooks\02_dynamic\fsl_dataset_clean.pt

DATASET
----------------------------------------------------------------------
X shape: (2129, 30, 126)
y shape: (2129,)
Samples: 2129
Unique labels: 105


MODEL: GREETING
Samples: 206
Input shape: (206, 30, 126)
Classes: 10

CLASS DISTRIBUTION
----------------------------------------------------------------------
  0 GOOD MORNING              20
  1 GOOD AFTERNOON            21
  2 GOOD EVENING              22
  3 HELLO                     20
  4 HOW ARE YOU               21
  5 IM FINE                   20
  6 NICE TO MEET YOU          22
  7 THANK YOU                 20
  8 YOURE WELCOME             20
  9 SEE YOU TOMORROW          20

LANDMARK QUALITY
------------------------------------------------------------------------------------------
GOOD MORNING              samples= 20 detect= 100.0% left=  97.0% right=   3.5

In [3]:
import torch

MODEL_PATH = r"C:\Projects\signia-fsl-recognition\artifacts\models\greeting\greetings_lstm_best.pt"

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint:
        print(" ", key)

    print("\nMetadata:")
    print("input_size:", checkpoint.get("input_size"))
    print("num_classes:", checkpoint.get("num_classes"))
    print("classes:", checkpoint.get("classes"))
    print("best_val_accuracy:", checkpoint.get("best_val_accuracy"))

Checkpoint type: <class 'dict'>

Checkpoint keys:
  model_state_dict
  input_size
  num_classes
  classes
  best_val_accuracy
  config

Metadata:
input_size: 126
num_classes: 10
classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
best_val_accuracy: 0.8064516129032258


In [4]:
import torch
import numpy as np

DATASET_PATH = r"C:\Projects\signia-fsl-recognition\notebooks\02_dynamic\fsl_dataset_clean.pt"

bundle = torch.load(
    DATASET_PATH,
    map_location="cpu",
    weights_only=False,
)

X = bundle["X"]
y = bundle["y"]

print("X shape:", X.shape)
print("X dtype:", X.dtype)

sample = X[0].numpy()

print("\nSample shape:", sample.shape)

print("Min:", sample.min())
print("Max:", sample.max())
print("Mean:", sample.mean())
print("Std:", sample.std())

print("\nFirst frame:")
print(sample[0])

print("\nFirst 10 features:")
print(sample[0][:10])

X shape: torch.Size([2129, 30, 126])
X dtype: torch.float32

Sample shape: (30, 126)
Min: -1.0
Max: 0.60986316
Mean: -0.049084365
Std: 0.19094463

First frame:
[ 0.          0.          0.          0.06597838 -0.2377441  -0.01017755
  0.14380792 -0.44407973 -0.0356174   0.19342555 -0.61843604 -0.06182826
  0.19822039 -0.79402107 -0.08770505  0.14882793 -0.32384056 -0.05023016
  0.21815117 -0.28940907 -0.08410301  0.2283408  -0.2713991  -0.11454148
  0.2213716  -0.25849614 -0.13720584  0.11781815 -0.13872671 -0.06942099
  0.19799669 -0.13705774 -0.08423737  0.21151386 -0.12760949 -0.09811886
  0.1965471  -0.1335806  -0.12109048  0.09334608  0.04272679 -0.08659986
  0.17228809  0.02307324 -0.10175284  0.18908873  0.03711911 -0.09269338
  0.17742862  0.02561607 -0.0955959   0.07445986  0.20597893 -0.10364774
  0.14495154  0.19487776 -0.11534335  0.16209808  0.1855116  -0.09805369
  0.15645252  0.1620865  -0.08945917  0.          0.          0.
  0.          0.          0.          0.     